# Supplement: Attention-Only Q/K Sparse — Teacher-Required Full Sparsity
## Q16 + Q8: 0 / 10 / 30 / 50 / 70 / 90 / 100

**SOFTWARE SIMULATION ONLY.** NO HLS. NO Vivado. NO PYNQ.

---
## Cell 1: Imports & Verification
---

In [ ]:
import sys, os, csv, numpy as np, time as time_module
from collections import OrderedDict
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print(f'Python: {sys.executable}')
assert 'anaconda' not in sys.executable.lower() and 'conda' not in sys.executable.lower()
print('OK: Docker kernel')

PROJ = '/home/cym/prj2/finn/notebooks/icl_thesis-master'
EXP  = f'{PROJ}/experiments/imgds_linear_sparse'
FP   = f'{EXP}/final_paper_experiments'
TABLES = f'{FP}/tables'
os.makedirs(TABLES, exist_ok=True)

if PROJ not in sys.path:
    sys.path.insert(0, PROJ)
from src.transformer import LinearUNSWAnomalyDetector, _UNSWLinearAttention
print('Imports OK')


---
## Cell 2: Constants & Model Wrapper
---

In [ ]:
D_MODEL = 16; SEQ_LEN = 16; INPUT_DIM = 64; FF_DIM = 32; N_CLASSES = 2

MODEL_CONFIG = {
    'input_dim': INPUT_DIM, 'seq_len': SEQ_LEN, 'd_model': D_MODEL,
    'dim_feedforward': FF_DIM, 'num_layers': 1, 'num_classes': N_CLASSES, 'dropout': 0.0,
}

# Wrapper for checkpoint with 'backbone.' prefix
class LinearIMGDSDenseBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = LinearUNSWAnomalyDetector(**MODEL_CONFIG)
    def forward(self, x):
        return self.backbone(x)

print('Model wrapper ready')


---
## Cell 3: Load Checkpoint & Test Data
---

In [ ]:
CKPT_PATH = f'{EXP}/checkpoints/best_linear_imgds_dense_r32_p8.pt'
ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
sd = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt

# Check for backbone. prefix
has_prefix = any(k.startswith('backbone.') for k in sd.keys())
print(f'Has backbone. prefix: {has_prefix}')

if has_prefix:
    sd_clean = OrderedDict()
    for k, v in sd.items():
        sd_clean[k.replace('backbone.', '')] = v
    sd_ref = sd
else:
    sd_clean = sd
    sd_ref = sd

# Verify loading
test_model = LinearIMGDSDenseBaseline()
test_model.backbone.load_state_dict(sd_clean, strict=True)
test_model.eval()
print(f'Model loaded OK. Params: {sum(p.numel() for p in test_model.backbone.parameters())}')

# Test data
X_test = np.load(f'{EXP}/outputs/imgds_r32_p8_test.npz')['X']
y_test = np.load(f'{EXP}/outputs/imgds_r32_p8_test.npz')['y']
print(f'Test: X={X_test.shape}, y={y_test.shape}, labels={np.bincount(y_test)}')


---
## Cell 4: Q/K Channel Importance
---

In [ ]:
def qk_importance(sd_dict):
    imp = np.zeros(D_MODEL)
    for prefix in ['layers.0.attention.query.weight', 'layers.0.attention.key.weight']:
        for k in sd_dict:
            if prefix in k and 'weight' in k:
                w = sd_dict[k].numpy()
                imp += np.abs(w).sum(axis=1)
                break
    return imp

# Try both prefixed and clean
imp = qk_importance(sd_ref)
if imp.sum() == 0:
    imp = qk_importance(sd_clean)
print(f'Q/K importance: {imp}')
sorted_rank = np.argsort(imp)[::-1]

def active_channels_for(sp_pct):
    mapping = {0:16, 10:14, 30:11, 50:8, 70:5, 90:2, 100:0}
    ad = mapping.get(sp_pct, max(round(D_MODEL*(1-sp_pct/100)), 0))
    if ad >= D_MODEL: return list(range(D_MODEL)), D_MODEL
    if ad <= 0: return [], 0
    return sorted(sorted_rank[:ad].tolist()), ad

for sp in [0,10,30,50,70,90,100]:
    ch, ad = active_channels_for(sp)
    print(f'  S={sp:3d}% -> ad={ad:2d} -> {ch}')


---
## Cell 5: Fake Quantization
---

In [ ]:
def fake_quant(w, bits, intb):
    if bits == 32: return w.clone()
    s = 2**(bits-intb)
    mx = 2**(intb-1)-2**(-(bits-intb))
    mn = -2**(intb-1)
    return torch.clamp(torch.round(w*s), mn*s, mx*s)/s

def quant_sd(sd_in, bits):
    if bits == 32:
        return OrderedDict((k, v.clone()) for k, v in sd_in.items())
    intb = {16:6, 8:4, 4:2}
    q = OrderedDict()
    for k, v in sd_in.items():
        is_w = 'weight' in k and v.ndim >= 2
        is_n = 'norm' in k.lower()
        q[k] = fake_quant(v, bits, intb[bits]) if (is_w and not is_n) else v.clone()
    return q

q16_sd = quant_sd(sd_clean, 16)
q8_sd  = quant_sd(sd_clean, 8)
q32_sd = OrderedDict((k, v.clone()) for k, v in sd_clean.items())
print('Quantized state_dicts ready')


---
## Cell 6: Attention-Only Q/K Sparse Monkey-Patch
---

In [ ]:
def make_sparse_forward(orig_fwd, active_ch, dm=16):
    mask = torch.zeros(dm)
    for c in active_ch:
        mask[c] = 1.0
    def sparse_forward(self, x):
        q = self._feature_map(self.query(x))
        k = self._feature_map(self.key(x))
        v = self.value(x)
        m = mask.to(x.device)
        qs = q * m[None, None, :]
        ks = k * m[None, None, :]
        kv = torch.matmul(ks.transpose(-2, -1), v)
        ks2 = ks.sum(dim=1, keepdim=False).unsqueeze(-1)
        nz = torch.matmul(qs, ks2).clamp_min(self.eps)
        return self.output(torch.matmul(qs, kv) / nz)
    return sparse_forward

def run_inference(model, X, active_ch, bs=512):
    backbone_mod = model.backbone if hasattr(model, 'backbone') else model
    attn = backbone_mod.layers[0].attention
    orig = attn.forward
    attn.forward = make_sparse_forward(orig, active_ch).__get__(attn, type(attn))
    try:
        logs, prs = [], []
        with torch.no_grad():
            for i in range(0, len(X), bs):
                b = torch.from_numpy(X[i:i+bs]).float()
                lo = model(b)
                logs.append(lo.numpy())
                prs.append(torch.argmax(lo, dim=1).numpy())
        return np.concatenate(logs, 0), np.concatenate(prs, 0)
    finally:
        attn.forward = orig

# Sanity check
full16 = list(range(D_MODEL))
model_s = LinearIMGDSDenseBaseline()
model_s.backbone.load_state_dict(q32_sd, strict=True)
model_s.eval()
logs_s, prs_s = run_inference(model_s, X_test[:100], full16)
print(f'Sanity: 100 samples, preds={np.bincount(prs_s)}, shape={logs_s.shape}')
print('Monkey-patch OK')


---
## Cell 7: Run Supplement — Q32 Baseline
---

In [ ]:
# Q32 baseline (needed as reference for match rate & logit error)
model_q32 = LinearIMGDSDenseBaseline()
model_q32.backbone.load_state_dict(q32_sd, strict=True)
model_q32.eval()
logits_q32, preds_q32 = run_inference(model_q32, X_test, full16)
q32_acc = accuracy_score(y_test, preds_q32)
print(f'Q32 baseline: acc={q32_acc:.4f}, preds={np.bincount(preds_q32)}')


---
## Cell 8: Run Supplement — Q16 & Q8 Full Sparsity
---

In [ ]:
TEACHER_SP = [0, 10, 30, 50, 70, 90, 100]
results = []

for qbits in [16, 8]:
    sd_use = q16_sd if qbits == 16 else q8_sd
    for sp in TEACHER_SP:
        active_ch, active_dim = active_channels_for(sp)
        eid = f'e1_q{qbits}_s{sp}_attn_only'
        t0 = time_module.time()

        model = LinearIMGDSDenseBaseline()
        model.backbone.load_state_dict(sd_use, strict=True)
        model.eval()

        logs, prs = run_inference(model, X_test, active_ch)
        acc = accuracy_score(y_test, prs)
        prec = precision_score(y_test, prs, zero_division=0)
        rec = recall_score(y_test, prs, zero_division=0)
        f1v = f1_score(y_test, prs, zero_division=0)
        probs = torch.softmax(torch.from_numpy(logs), dim=1).numpy()
        auc = roc_auc_score(y_test, probs[:, 1]) if len(np.unique(y_test)) > 1 else 0.5
        match = (prs == np.argmax(logits_q32, axis=1)).mean()
        lerr = np.abs(logs - logits_q32)
        dt = time_module.time() - t0

        note = 'high-sparsity ablation' if sp >= 50 else ''
        r = {
            'experiment_id': eid,
            'quant_bits': qbits,
            'sparsity_percent': sp,
            'active_dim': active_dim,
            'active_channels': str(active_ch),
            'accuracy': round(acc, 6),
            'precision': round(prec, 6),
            'recall': round(rec, 6),
            'f1': round(f1v, 6),
            'auc': round(auc, 6),
            'prediction_match_rate_vs_q32': round(match, 6),
            'max_logit_error_vs_q32': round(lerr.max(), 6),
            'mean_logit_error_vs_q32': round(lerr.mean(), 6),
            'sparse_type': 'attention_only_qk_active_channel',
            'status': 'SUCCESS',
            'notes': note,
        }
        results.append(r)

        acc_pct = acc * 100
        tag = ' [ABLATION]' if sp >= 50 else ''
        vs = ''
        if sp < 50:
            vs = f' vs ViT4Mal: {"ABOVE" if acc_pct >= 92.41 else "BELOW"}'
        print(f'  [{eid}] S={sp:3d}% ad={active_dim:2d} acc={acc_pct:7.2f}% match={match:.4f} ({dt:.1f}s){tag}{vs}')

n_total = len(results)
n_above = sum(1 for r in results if r['accuracy'] >= 0.9241)
print(f'\nAll {n_total} done. {n_above} above ViT4Mal fastest (92.41%)')


---
## Cell 9: Save Results
---

In [ ]:
out_csv = f'{TABLES}/experiment1_attention_only_full_sparsity_software_results.csv'
fnames = ['experiment_id','quant_bits','sparsity_percent','active_dim','active_channels',
          'accuracy','precision','recall','f1','auc','prediction_match_rate_vs_q32',
          'max_logit_error_vs_q32','mean_logit_error_vs_q32','sparse_type','status','notes']

with open(out_csv, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fnames, extrasaction='ignore')
    w.writeheader()
    w.writerows(results)

print(f'Saved: {out_csv}')

# Summary table
print(f'\n{"ID":<28} {"Q":<4} {"S%":<5} {"Ad":<5} {"Acc%":<8} {"Match":<8} {"AUC":<8} {"Notes"}')
print('-' * 90)
for r in results:
    acc_pct = r['accuracy'] * 100
    print(f'{r["experiment_id"]:<28} {r["quant_bits"]:<4} {r["sparsity_percent"]:<5} '
          f'{r["active_dim"]:<5} {acc_pct:<8.2f} {r["prediction_match_rate_vs_q32"]:<8.4f} '
          f'{r["auc"]:<8.4f} {r["notes"]}')

# Key findings
q16_dense = [r for r in results if r['experiment_id']=='e1_q16_s0_attn_only'][0]
print(f'\nKey findings:')
print(f'  Q16 S0 (dense): acc={q16_dense["accuracy"]*100:.2f}%')
for sp in [10,30,50,70,90,100]:
    pts = [r for r in results if r['experiment_id']==f'e1_q16_s{sp}_attn_only']
    if pts:
        a = pts[0]['accuracy']*100
        drop = (q16_dense['accuracy'] - pts[0]['accuracy'])*100
        print(f'  Q16 S{sp}: acc={a:.2f}% (drop={drop:.2f}pp)')
print('\nSupplement complete.')
